# RTMDet-tiny reproduction


In [1]:
import subprocess

gpu = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total",
        "--format=csv,noheader",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(gpu)
if not gpu.startswith("Tesla T4,"):
    raise RuntimeError("select a Tesla T4 runtime for this reproduction")

Tesla T4, 580.82.07, 15360 MiB


In [2]:
import os
from pathlib import Path

installer = Path("/tmp/uv-install.sh")
subprocess.run(
    [
        "curl",
        "-LsSf",
        "https://astral.sh/uv/0.12.23/install.sh",
        "-o",
        str(installer),
    ],
    check=True,
)
env = os.environ.copy()
env["UV_UNMANAGED_INSTALL"] = "/usr/local/bin"
subprocess.run(["sh", str(installer)], check=True, env=env)

uv_version = subprocess.run(
    ["uv", "--version"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(uv_version)
if not uv_version.startswith("uv 0.12.23 "):
    raise RuntimeError(f"unexpected uv version: {uv_version}")

uv 0.12.23 (x86_64-unknown-linux-gnu)


In [3]:
import shutil

repository = "https://github.com/aekn/smallpotato.git"
revision = "071a4a66acd9e8b3ae3b523110a6becfe208b304"
repo = Path("/content/smallpotato")

if repo.exists():
    shutil.rmtree(repo)

subprocess.run(["git", "init", "-q", str(repo)], check=True)
subprocess.run(
    ["git", "-C", str(repo), "remote", "add", "origin", repository],
    check=True,
)
subprocess.run(
    [
        "git",
        "-C",
        str(repo),
        "fetch",
        "-q",
        "--depth=1",
        "--no-tags",
        "origin",
        revision,
    ],
    check=True,
)
subprocess.run(
    ["git", "-C", str(repo), "checkout", "-q", "--detach", "FETCH_HEAD"],
    check=True,
)

actual = subprocess.run(
    ["git", "-C", str(repo), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if actual != revision:
    raise RuntimeError(f"expected {revision}, got {actual}")

status = subprocess.run(
    ["git", "-C", str(repo), "status", "--porcelain"],
    check=True,
    capture_output=True,
    text=True,
).stdout
if status:
    raise RuntimeError("repository checkout is dirty")

os.chdir(repo)
print(f"{repo} @ {actual}")

/content/smallpotato @ 071a4a66acd9e8b3ae3b523110a6becfe208b304


In [4]:
subprocess.run(["uv", "sync", "--locked"], cwd=repo, check=True)
print("environment ready")

environment ready


In [5]:
process = subprocess.Popen(
    ["uv", "run", "python", "-m", "reproductions.rtmdet.reproduce"],
    cwd=repo,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)

returncode = process.wait()
if returncode:
    raise subprocess.CalledProcessError(returncode, process.args)

smallpotato - reproduce rtmdet

model       rtmdet-tiny
dataset     coco/val2017
run         /content/smallpotato/runs/rtmdet/20261009T213347Z

prepare     inputs
check       source                      ok
check       data                        ok
check       checkpoint                  ok
sync        environment
device      Tesla T4
check       environment                 ok
test          50/1000    5.0%   eta 6m22s
test         100/1000   10.0%   eta 5m55s
test         150/1000   15.0%   eta 5m37s
test         200/1000   20.0%   eta 5m13s
test         250/1000   25.0%   eta 4m54s
test         300/1000   30.0%   eta 4m33s
test         350/1000   35.0%   eta 4m13s
test         400/1000   40.0%   eta 3m53s
test         450/1000   45.0%   eta 3m33s
test         500/1000   50.0%   eta 3m14s
test         550/1000   55.0%   eta 2m54s
test         600/1000   60.0%   eta 2m35s
test         650/1000   65.0%   eta 2m15s
test         700/1000   70.0%   eta 1m56s
test         750/1000   75.0%   

In [6]:
runs = repo / "runs" / "rtmdet"
run_dir = max(
    (path for path in runs.iterdir() if path.is_dir()),
    key=lambda path: path.name,
)
print(run_dir)

for name in ("summary.json", "provenance.json"):
    print(f"\n{name}")
    print((run_dir / name).read_text(encoding="utf-8"), end="")

/content/smallpotato/runs/rtmdet/20261009T213347Z

summary.json
{
  "durations_seconds": {
    "environment": 81.70321000299998,
    "evaluation": 140.45151407700007,
    "inference": 438.67323520900004,
    "inputs": 48.12456919599998,
    "total": 709.328588735
  },
  "finished_at": "2026-10-09T21:45:36.629363Z",
  "metrics": {
    "ap": 0.41120437947983723,
    "ap50": 0.5786713136975927,
    "ap75": 0.4472345235150815,
    "ap_large": 0.5830372446759329,
    "ap_medium": 0.4550520685210053,
    "ap_small": 0.209897601988187,
    "ar1": 0.3339977181430133,
    "ar10": 0.5535840902808274,
    "ar100": 0.6052522448534676,
    "ar_large": 0.796902703107174,
    "ar_medium": 0.6768884067153174,
    "ar_small": 0.38072385922342455
  },
  "prediction": {
    "path": "pred.bbox.json",
    "sha256": "1dc864d877662a6544f6fe218862a59e3f7142a0c3c17464dde9910e1372ae83",
    "size": 234092583
  },
  "reference": {
    "max_abs_error": 4.6532333541904336e-11,
    "metrics": {
      "ap": 0.411204

In [7]:
from google.colab import files

archive_base = Path("/content") / f"rtmdet-{run_dir.name}"
run_archive = shutil.make_archive(
    str(archive_base),
    "gztar",
    root_dir=run_dir.parent,
    base_dir=run_dir.name,
)
print(run_archive)
files.download(run_archive)

/content/rtmdet-20261009T213347Z.tar.gz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>